In [ ]:
import os
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.models as models
import pytorch_lightning as pl
from PIL import Image
from lightning.pytorch import seed_everything
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, random_split
from torchmetrics.classification import F1Score, MulticlassConfusionMatrix
from torchvision import transforms

from platformtools.storage import PlatformStorage

In [ ]:
# The dataset to train on, and where inside the ARC its labeled images live.
DATASET_ID = 1
IMAGES_SUBPATH = "assays/drone_image_capture/dataset/images/labeled"

RANDOM_SEED = 3012
seed_everything(RANDOM_SEED, workers=True)

class_to_idx = {
    "negative": 0,
    "positive": 1,
}

In [ ]:
# Mount the dataset's bucket and resolve the local path to its labeled images.
s3 = PlatformStorage()
datasets = s3.datasets.list()
dataset_info = next(d for d in datasets if d["id"] == DATASET_ID)

s3.mount_bucket_for_object(dataset_info)
dataset_root = s3.get_mounted_path_for_object(dataset_info)
images_root = os.path.join(dataset_root, IMAGES_SUBPATH)
images_root

In [ ]:
def image_loader(path):
    return Image.open(path).convert("RGB")


def is_image_file(path):
    return path.lower().endswith((".jpg", ".jpeg", ".png"))


def label_for_path(path: Path) -> int:
    haystack = str(path).lower()
    is_negative = "_negativ" in haystack
    is_positive = "_positiv" in haystack
    if is_negative == is_positive:
        raise ValueError(
            f"Could not determine a unique label for '{path}' "
            "(expected exactly one of '_negativ' / '_positiv' in its path)."
        )
    return class_to_idx["positive" if is_positive else "negative"]


class LabeledImageDataset(Dataset):
    """Images labeled by a '_negativ'/'_positiv' marker in their path, as
    found under an ARC's assays/.../images/labeled folder mounted from MinIO.
    """

    def __init__(self, root, transform=None):
        self.transform = transform
        self.classes = list(class_to_idx.keys())
        self.class_to_idx = class_to_idx
        self.samples = [
            (str(path), label_for_path(path))
            for path in sorted(Path(root).rglob("*"))
            if path.is_file() and is_image_file(str(path))
        ]
        if not self.samples:
            raise RuntimeError(f"No labeled images found under {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = image_loader(path)
        if self.transform:
            image = self.transform(image)
        return image, label


def count_samples_per_class(dataset):
    counter = Counter()
    for _, label in dataset.samples:
        counter[label] += 1

    print("Samples per class:")
    for cls_name in dataset.classes:
        idx = dataset.class_to_idx[cls_name]
        print(f"  {cls_name} [{idx}]: {counter[idx]} samples")

In [ ]:
class ImageDataModule(pl.LightningDataModule):
    def __init__(self, dataset, batch_size=32, num_workers=0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.train_dataset, self.val_dataset, self.test_dataset = random_split(
            dataset,
            [0.7, 0.1, 0.2]
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

In [ ]:
dataset = LabeledImageDataset(
    root=images_root,
    transform=transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
    ])
)
count_samples_per_class(dataset)

# num_workers=0 avoids DataLoader worker subprocesses, since forking after
# CUDA has been initialized (as the Trainer below does) can deadlock them
# indefinitely in a notebook, with no error and no further output.
dm = ImageDataModule(
    dataset=dataset,
    batch_size=16,
    num_workers=0
)

In [ ]:
class AgdafairModule(pl.LightningModule):
    def __init__(self, num_classes, model):
        super().__init__()
        self.f1_train = F1Score(task="multiclass", num_classes=num_classes)
        self.f1_val = F1Score(task="multiclass", num_classes=num_classes)
        self.f1_test = F1Score(task="multiclass", num_classes=num_classes)
        self.metrics_train = MulticlassConfusionMatrix(num_classes=2)
        self.metrics_val = MulticlassConfusionMatrix(num_classes=2)
        self.metrics_test = MulticlassConfusionMatrix(num_classes=2)
        self.model = model

    def forward(self, inputs):
        return self.model(inputs)

    def training_step(self, batch, batch_idx):
        inputs, target = batch
        output = self(inputs)
        self.f1_train.update(output, target)
        #self.metrics_train.update(output, target)
        loss = torch.nn.functional.cross_entropy(output, target)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def on_training_epoch_end(self):
        f1_epoch = self.f1_train.compute()
        self.log("train_f1", f1_epoch, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.f1_val.reset()
        #metrics_epoch, _ = self.metrics_train.plot()
        self.metrics_train.reset()

    def validation_step(self, batch, batch_idx):
        inputs, target = batch
        output = self(inputs)
        self.f1_val.update(output, target)
        #self.metrics_val.update(output, target)
        loss = torch.nn.functional.cross_entropy(output, target)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def on_validation_epoch_end(self):
        f1_epoch = self.f1_val.compute()
        self.log("val_f1", f1_epoch, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.f1_val.reset()
        #metrics_epoch, _ = self.metrics_val.plot()
        self.metrics_val.reset()

    def test_step(self, batch, batch_idx):
        inputs, target = batch
        output = self(inputs)
        self.f1_test.update(output, target)
        self.metrics_test.update(output, target)
        loss = torch.nn.functional.cross_entropy(output, target)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def on_test_epoch_end(self):
        f1_epoch = self.f1_test.compute()
        self.log("test_f1", f1_epoch, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.f1_test.reset()
        metrics_epoch, _ = self.metrics_test.plot()
        self.metrics_test.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.model.parameters())
        scheduler = ReduceLROnPlateau(optimizer, mode="min")
        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
            "monitor": "val_loss",
        }

In [ ]:
num_classes = 2
resnet101 = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
num_features = resnet101.fc.in_features
resnet101.fc = nn.Linear(num_features, num_classes)

module = AgdafairModule(num_classes=2, model=resnet101)
trainer = pl.Trainer(max_epochs=100, devices=1, accelerator="gpu", log_every_n_steps=1)
trainer.fit(model=module, datamodule=dm)

In [ ]:
# automatically loads the best weights for you
trainer.test(model=module, datamodule=dm)

In [ ]:
module.to_torchscript(file_path="./virus_classificator.pt")